In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
# ── Génération du dataset ───────────────────────────────────────────
n = 80
marches = ['Mokolo','Sandaga','Marché Mvog-Mbi','New-Bell','Mfoundi']
produits = ['Riz (1kg)','Huile (1L)','Tomate (kg)','Plantain (kg)','Poulet (kg)','Igname (kg)','Arachide (kg)','Poisson fumé (kg)']
mois = ['Janvier 2024','Février 2024','Mars 2024']

# Prix de référence par produit (FCFA)
prix_ref = {
    'Riz (1kg)':500,'Huile (1L)':1200,'Tomate (kg)':250,
    'Plantain (kg)':300,'Poulet (kg)':2500,'Igname (kg)':400,
    'Arachide (kg)':600,'Poisson fumé (kg)':3500
}
data = []


for i in range(n):
    prod = np.random.choice(produits)
    marche = np.random.choice(marches)
    mois_v = np.random.choice(mois)
    prix_b = prix_ref[prod]
    # Prix avec variation réaliste (±20%)
    prix = max(50, int(prix_b * np.random.uniform(0.8, 1.2)))
    qty = np.random.randint(1, 50)
    data.append({'mois':mois_v,'marche':marche,'produit':prod,'prix_fcfa':prix,'quantite':qty,'total':prix*qty})


df_raw = pd.DataFrame(data)

# ── Introduire des problèmes intentionnels ──────────────────────────
idx_nan = np.random.choice(df_raw.index, 8, replace=False)
idx_dup = np.random.choice(df_raw.index, 5, replace=False)
idx_out = np.random.choice(df_raw.index, 4, replace=False)

# Valeurs manquantes
df_raw.loc[idx_nan[:4], 'prix_fcfa'] = np.nan
df_raw.loc[idx_nan[4:], 'quantite'] = np.nan

# Doublons
df_raw = pd.concat([df_raw, df_raw.loc[idx_dup]], ignore_index=True)

# Outliers dans prix_fcfa
df_raw.loc[idx_out[:2], 'prix_fcfa'] = df_raw.loc[idx_out[:2], 'prix_fcfa'] * 10
df_raw.loc[idx_out[2:], 'quantite'] = 0

# Incohérences de casse dans marche
df_raw.loc[df_raw.index[:10], 'marche'] = df_raw.loc[df_raw.index[:10], 'marche'].str.upper()

print(df_raw)
print(f'Dataset brut : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes')

print(df_raw.head(10))
print(df_raw.tail(10))


            mois           marche            produit  prix_fcfa  quantite  \
0   Janvier 2024         NEW-BELL      Arachide (kg)      524.0       8.0   
1      Mars 2024          MFOUNDI        Poulet (kg)     2445.0      23.0   
2   Janvier 2024  MARCHÉ MVOG-MBI        Tomate (kg)      260.0      24.0   
3   Février 2024          MFOUNDI        Tomate (kg)      272.0      30.0   
4   Janvier 2024          SANDAGA        Igname (kg)      418.0      22.0   
..           ...              ...                ...        ...       ...   
80  Janvier 2024         New-Bell         Huile (1L)     1056.0      32.0   
81  Février 2024          Sandaga        Igname (kg)      325.0      18.0   
82  Janvier 2024           Mokolo      Plantain (kg)      299.0      48.0   
83  Février 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3106.0      24.0   
84  Février 2024          Sandaga         Huile (1L)     1175.0      41.0   

    total  
0    4192  
1   56235  
2    6240  
3    8160  
4    9196  
.. 

ce code affiche le contenu du dataset, ses dimension du dataset et les 10 premières lignes du dataset. La nature de celui-ci est : un objet

In [5]:
df_raw.shape

(85, 6)

In [7]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   mois       85 non-null     object 
 1   marche     85 non-null     object 
 2   produit    85 non-null     object 
 3   prix_fcfa  81 non-null     float64
 4   quantite   81 non-null     float64
 5   total      85 non-null     int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 4.1+ KB


In [8]:
print(df_raw.isnull().sum())

mois         0
marche       0
produit      0
prix_fcfa    4
quantite     4
total        0
dtype: int64


In [9]:
print(f'le nombre totale de doublons est de : \n',df_raw.duplicated().sum())
print(df_raw.duplicated())

le nombre totale de doublons est de : 
 5
0     False
1     False
2     False
3     False
4     False
      ...  
80     True
81     True
82     True
83     True
84     True
Length: 85, dtype: bool


In [10]:
print(df_raw.describe())

         prix_fcfa   quantite          total
count    81.000000  81.000000      85.000000
mean   1245.086420  24.938272   29279.364706
std    1288.815388  14.390401   35707.505157
min     228.000000   0.000000     632.000000
25%     368.000000  14.000000    6993.000000
50%     564.000000  24.000000   12672.000000
75%    1399.000000  35.000000   33792.000000
max    6740.000000  49.000000  161586.000000


Problème	        Colonne concernée	Nb de cas	Action prévue
Valeurs manquantes	prix_fcfa	               4	Médiane produit
Valeurs manquantes	quantite	               4	Médiane produit
Doublons exacts	    Toutes colonnes	           5   drop_duplicates()
Outliers prix	    prix_fcfa	              14	IQR ou clip()
Quantités = 0	    quantite	               0	Remplacer par                                                       NaN puis médiane
Casse incohérente	marche	                   5	str.title()


In [12]:
df_raw['prix_fcfa'] = df_raw['prix_fcfa'].fillna(df_raw['prix_fcfa'].median())
print(df_raw['prix_fcfa'])
print(df_raw.tail(10))

0      524.0
1     2445.0
2      260.0
3      272.0
4      418.0
       ...  
80    1056.0
81     325.0
82     299.0
83    3106.0
84    1175.0
Name: prix_fcfa, Length: 85, dtype: float64
            mois           marche            produit  prix_fcfa  quantite  \
75     Mars 2024         New-Bell  Poisson fumé (kg)     3273.0      28.0   
76     Mars 2024          Sandaga         Huile (1L)     1282.0      25.0   
77  Février 2024          Mfoundi         Huile (1L)      564.0      22.0   
78  Février 2024         New-Bell          Riz (1kg)      564.0      43.0   
79  Janvier 2024          Mfoundi      Plantain (kg)      339.0      44.0   
80  Janvier 2024         New-Bell         Huile (1L)     1056.0      32.0   
81  Février 2024          Sandaga        Igname (kg)      325.0      18.0   
82  Janvier 2024           Mokolo      Plantain (kg)      299.0      48.0   
83  Février 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3106.0      24.0   
84  Février 2024          Sandaga         H

In [13]:
df_raw['quantite'] = df_raw['quantite'].fillna(df_raw['quantite'].median())
print(df_raw['quantite'])
print(df_raw.head(10))

0      8.0
1     23.0
2     24.0
3     30.0
4     22.0
      ... 
80    32.0
81    18.0
82    48.0
83    24.0
84    41.0
Name: quantite, Length: 85, dtype: float64
           mois           marche            produit  prix_fcfa  quantite  \
0  Janvier 2024         NEW-BELL      Arachide (kg)      524.0       8.0   
1     Mars 2024          MFOUNDI        Poulet (kg)     2445.0      23.0   
2  Janvier 2024  MARCHÉ MVOG-MBI        Tomate (kg)      260.0      24.0   
3  Février 2024          MFOUNDI        Tomate (kg)      272.0      30.0   
4  Janvier 2024          SANDAGA        Igname (kg)      418.0      22.0   
5  Janvier 2024         NEW-BELL        Poulet (kg)     2291.0      42.0   
6     Mars 2024         NEW-BELL  Poisson fumé (kg)     3438.0      47.0   
7     Mars 2024  MARCHÉ MVOG-MBI        Igname (kg)      477.0      24.0   
8     Mars 2024  MARCHÉ MVOG-MBI        Poulet (kg)     2170.0      39.0   
9  Janvier 2024         NEW-BELL         Huile (1L)     1423.0       9.0   


In [9]:
print(df_raw.drop_duplicates())

            mois           marche            produit  prix_fcfa  quantite  \
0   Janvier 2024         NEW-BELL      Arachide (kg)      524.0       8.0   
1      Mars 2024          MFOUNDI        Poulet (kg)     2445.0      23.0   
2   Janvier 2024  MARCHÉ MVOG-MBI        Tomate (kg)      260.0      24.0   
3   Février 2024          MFOUNDI        Tomate (kg)      272.0      30.0   
4   Janvier 2024          SANDAGA        Igname (kg)      418.0      22.0   
..           ...              ...                ...        ...       ...   
75     Mars 2024         New-Bell  Poisson fumé (kg)     3273.0      28.0   
76     Mars 2024          Sandaga         Huile (1L)     1282.0      25.0   
77  Février 2024          Mfoundi         Huile (1L)      564.0      22.0   
78  Février 2024         New-Bell          Riz (1kg)      564.0      43.0   
79  Janvier 2024          Mfoundi      Plantain (kg)      339.0      44.0   

    total  
0    4192  
1   56235  
2    6240  
3    8160  
4    9196  
.. 

In [5]:
Q1 = df_raw['prix_fcfa'].quantile(0.25)
Q3 = df_raw['prix_fcfa'].quantile(0.75)
IQR = Q3 - Q1

borneInf = Q1 - 1.5 * IQR
borneSup = Q3 + 1.5 * IQR

valeurA = df_raw[(df_raw['prix_fcfa'] < borneInf) | (df_raw['prix_fcfa'] > borneSup)]
print(valeurA)
print(f'{len(valeurA)} valeurs Aberrantes détectés')

            mois           marche            produit  prix_fcfa  quantite  \
6      Mars 2024         NEW-BELL  Poisson fumé (kg)     3438.0      47.0   
16  Février 2024          Mfoundi  Poisson fumé (kg)     3555.0      24.0   
18  Janvier 2024           Mokolo        Poulet (kg)     2986.0      24.0   
20     Mars 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3879.0       5.0   
36  Février 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3106.0      24.0   
39     Mars 2024           Mokolo  Poisson fumé (kg)     3653.0       2.0   
41  Février 2024         New-Bell        Poulet (kg)     2985.0      32.0   
45     Mars 2024          Mfoundi  Poisson fumé (kg)     3627.0      36.0   
47  Janvier 2024         New-Bell  Poisson fumé (kg)     3767.0      39.0   
59     Mars 2024  Marché Mvog-Mbi      Plantain (kg)     3160.0      22.0   
68  Février 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3579.0      31.0   
72  Février 2024          Mfoundi      Arachide (kg)     6740.0      35.0   

In [17]:
df_raw['marche'] = df_raw['marche'].str.title()
print("DataFrame après standardisation en Title Case:")
print(df_raw['marche'])

DataFrame après standardisation en Title Case:
0            New-Bell
1             Mfoundi
2     Marché Mvog-Mbi
3             Mfoundi
4             Sandaga
           ...       
80           New-Bell
81            Sandaga
82             Mokolo
83    Marché Mvog-Mbi
84            Sandaga
Name: marche, Length: 85, dtype: object


In [16]:
df = df_raw.copy()
print(f'Avant nettoyage : {df.shape[0]} lignes')

Avant nettoyage : 85 lignes


In [19]:
df['marche'] = df['marche'].str.title()
print("DataFrame après standardisation en Title Case:")
print(df)

print("Vérification de la cohérence des noms de marchés:")
print(df['marche'].value_counts())

DataFrame après standardisation en Title Case:
            mois           marche            produit  prix_fcfa  quantite  \
0   Janvier 2024         New-Bell      Arachide (kg)      524.0       8.0   
1      Mars 2024          Mfoundi        Poulet (kg)     2445.0      23.0   
2   Janvier 2024  Marché Mvog-Mbi        Tomate (kg)      260.0      24.0   
3   Février 2024          Mfoundi        Tomate (kg)      272.0      30.0   
4   Janvier 2024          Sandaga        Igname (kg)      418.0      22.0   
..           ...              ...                ...        ...       ...   
80  Janvier 2024         New-Bell         Huile (1L)     1056.0      32.0   
81  Février 2024          Sandaga        Igname (kg)      325.0      18.0   
82  Janvier 2024           Mokolo      Plantain (kg)      299.0      48.0   
83  Février 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3106.0      24.0   
84  Février 2024          Sandaga         Huile (1L)     1175.0      41.0   

    total  
0    4192  
1   

In [25]:
df_cleaned = df.drop_duplicates()

print("DataFrame après suppression des doublons:")
print(df_cleaned)

nombre_lignes_restantes = len(df_cleaned)
print(f"Il reste {nombre_lignes_restantes} lignes après la suppression des doublons.")

DataFrame après suppression des doublons:


,mois,marche,produit,prix_fcfa,quantite,total
0,Janvier 2024,New-Bell,Arachide (kg),524.00000,8.0,4192
1,Mars 2024,Mfoundi,Poulet (kg),2445.00000,23.0,56235
2,Janvier 2024,Marché Mvog-Mbi,Tomate (kg),260.00000,24.0,6240
3,Février 2024,Mfoundi,Tomate (kg),272.00000,30.0,8160
4,Janvier 2024,Sandaga,Igname (kg),418.00000,22.0,9196
...,...,...,...,...,...,...
75,Mars 2024,New-Bell,Poisson fumé (kg),3273.00000,28.0,91644
76,Mars 2024,Sandaga,Huile (1L),1282.00000,25.0,32050
77,Février 2024,Mfoundi,Huile (1L),1245.08642,22.0,27346
78,Février 2024,New-Bell,Riz (1kg),564.00000,43.0,24252


Il reste 80 lignes après la suppression des doublons.


In [20]:
mediane_prix = df.groupby('produit')['prix_fcfa'].transform('median')
df['prix_fcfa'] = df['prix_fcfa'].fillna(mediane_prix)

mediane_qty = df.groupby('produit')['quantite'].transform('median')
df['quantite'] = df['quantite'].fillna(mediane_qty)
 
print(f"le DataFrame {df} après imputation des valeurs manquantes:")
print(df)
print('NaN restants :', df.isnull().sum().sum())

le DataFrame             mois           marche            produit  prix_fcfa  quantite  \
0   Janvier 2024         New-Bell      Arachide (kg)      524.0       8.0   
1      Mars 2024          Mfoundi        Poulet (kg)     2445.0      23.0   
2   Janvier 2024  Marché Mvog-Mbi        Tomate (kg)      260.0      24.0   
3   Février 2024          Mfoundi        Tomate (kg)      272.0      30.0   
4   Janvier 2024          Sandaga        Igname (kg)      418.0      22.0   
..           ...              ...                ...        ...       ...   
80  Janvier 2024         New-Bell         Huile (1L)     1056.0      32.0   
81  Février 2024          Sandaga        Igname (kg)      325.0      18.0   
82  Janvier 2024           Mokolo      Plantain (kg)      299.0      48.0   
83  Février 2024  Marché Mvog-Mbi  Poisson fumé (kg)     3106.0      24.0   
84  Février 2024          Sandaga         Huile (1L)     1175.0      41.0   

    total  
0    4192  
1   56235  
2    6240  
3    8160  
4 

In [27]:
def corriger_outliers_iqr(serie, nom_col=''):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR
    n_out = ((serie < borne_inf) | (serie > borne_sup)).sum()
    print(f'{nom_col}: {n_out} outliers → [{borne_inf:.0f}, {borne_sup:.0f}]')
    return serie.clip(lower=borne_inf, upper=borne_sup)


df['prix_fcfa'] = corriger_outliers_iqr(df['prix_fcfa'], 'prix_fcfa')
df['quantite'] = corriger_outliers_iqr(df['quantite'], 'quantite')

df['total'] = df['prix_fcfa'] * df['quantite']

print(f'Après nettoyage complet : {df.shape[0]} lignes, {df.isnull().sum().sum()} NaN')
print(df)

prix_fcfa: 16 outliers → [-896, 2588]
quantite: 0 outliers → [-15, 65]
Après nettoyage complet : 85 lignes, 0 NaN


,mois,marche,produit,prix_fcfa,quantite,total
0,Janvier 2024,New-Bell,Arachide (kg),524.0,8.0,4192.0
1,Mars 2024,Mfoundi,Poulet (kg),2445.0,23.0,56235.0
2,Janvier 2024,Marché Mvog-Mbi,Tomate (kg),260.0,24.0,6240.0
3,Février 2024,Mfoundi,Tomate (kg),272.0,30.0,8160.0
4,Janvier 2024,Sandaga,Igname (kg),418.0,22.0,9196.0
...,...,...,...,...,...,...
80,Janvier 2024,New-Bell,Huile (1L),1056.0,32.0,33792.0
81,Février 2024,Sandaga,Igname (kg),325.0,18.0,5850.0
82,Janvier 2024,Mokolo,Plantain (kg),299.0,48.0,14352.0
83,Février 2024,Marché Mvog-Mbi,Poisson fumé (kg),2588.5,24.0,62124.0


In [26]:
classement_produits = df.groupby('produit')['prix_fcfa'].mean().sort_values(ascending=False)

print("\nClassement des produits par prix moyen (du plus cher au moins cher):")
display(classement_produits)

produit_plus_cher = classement_produits.index[0]
prix_moyen_plus_cher = classement_produits.iloc[0]
produit_moins_cher = classement_produits.index[-1]
prix_moyen_moins_cher = classement_produits.iloc[-1]

print(f"\nLe produit le plus cher en moyenne est '{produit_plus_cher}' avec un prix moyen de {prix_moyen_plus_cher:.2f} FCFA.")
print(f"Le produit le moins cher en moyenne est '{produit_moins_cher}' avec un prix moyen de {prix_moyen_moins_cher:.2f} FCFA.")


Classement des produits par prix moyen (du plus cher au moins cher):


produit
Poisson fumé (kg)    3446.909091
Poulet (kg)          2313.714286
Arachide (kg)        1362.625000
Huile (1L)           1090.777778
Plantain (kg)         602.636364
Riz (1kg)             490.444444
Igname (kg)           409.153846
Tomate (kg)           262.375000
Name: prix_fcfa, dtype: float64


Le produit le plus cher en moyenne est 'Poisson fumé (kg)' avec un prix moyen de 3446.91 FCFA.
Le produit le moins cher en moyenne est 'Tomate (kg)' avec un prix moyen de 262.38 FCFA.


In [29]:
# Calcul du chiffre d'affaires total par produit
chiffre_affaires_produit = df.groupby('produit')['total'].sum().sort_values(ascending=False)

print("Chiffre d'affaires total par produit:")
display(chiffre_affaires_produit)

# Produit avec le chiffre d'affaires total le plus élevé
produit_max_ca = chiffre_affaires_produit.index[0]
valeur_max_ca = chiffre_affaires_produit.iloc[0]

print(f"\nLe produit avec le chiffre d'affaires total le plus élevé est '{produit_max_ca}' avec un total de {valeur_max_ca:.2f} FCFA.")

Chiffre d'affaires total par produit:


produit
Poisson fumé (kg)    732545.500000
Huile (1L)           554791.395062
Poulet (kg)          384533.172840
Plantain (kg)        193258.148148
Arachide (kg)        176971.500000
Igname (kg)          111345.555556
Riz (1kg)             90549.222222
Tomate (kg)           61921.000000
Name: total, dtype: float64


Le produit avec le chiffre d'affaires total le plus élevé est 'Poisson fumé (kg)' avec un total de 732545.50 FCFA.
